# Common Crawl Collection Reporting

This notebook collates the small set of Common Crawl collection numbers that are useful for the Materials section of the dissertation. It reads the configured crawl years, final processed trend output, accepted corpus quality summaries, processed corpus document file, and throughput summaries.

The only saved output is `reports/tables/commoncrawl_collection_reporting.csv`. The notebook still performs consistency checks before writing that file, so stale or incomplete synced artifacts fail loudly rather than producing misleading paper numbers.

Corpus target counts distinguish unique target documents from non-exclusive concept membership. A document can mention both ADHD and autism; such documents are counted once in `target_unique_documents`, once in each relevant membership column, and explicitly in `adhd_autism_overlap_documents`.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

REPO_ROOT = Path.cwd().resolve()
for candidate_root in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate_root / "configs/commoncrawl_collection.yaml").exists():
        REPO_ROOT = candidate_root
        break
else:
    raise FileNotFoundError("Could not find configs/commoncrawl_collection.yaml in the current path or its parents.")

CONFIG_PATH = REPO_ROOT / "configs/commoncrawl_collection.yaml"
INTERIM_COLLECTION_DIR = REPO_ROOT / "data/interim/collection"
PROCESSED_DIR = REPO_ROOT / "data/processed"
REPORTING_CSV_PATH = REPO_ROOT / "reports/tables/summary/commoncrawl_collection_reporting.csv"

## 1. Reporting Frame

The configured crawl map defines the expected reporting frame. The instance type is recorded from the public config so throughput can be described in relation to the machine used for collection.

In [ ]:
with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

crawl_map = pd.DataFrame(config["collection"]["crawl_map"])
configured_years = sorted(crawl_map["year"].astype(int).tolist())
year_min = min(configured_years)
year_max = max(configured_years)
year_count = len(configured_years)
instance_type = config["collection"].get("aws", {}).get("ec2", {}).get("instance_type", "unknown")

crawl_map

,year,crawl_id
0,2014,CC-MAIN-2014-15
1,2015,CC-MAIN-2015-18
2,2016,CC-MAIN-2016-18
3,2017,CC-MAIN-2017-17
4,2018,CC-MAIN-2018-17
5,2019,CC-MAIN-2019-18
6,2020,CC-MAIN-2020-16
7,2021,CC-MAIN-2021-17
8,2022,CC-MAIN-2022-21
9,2023,CC-MAIN-2023-14


## 2. Small Helpers

These helpers keep the notebook compact while preserving the main reporting logic in plain, inspectable cells below.

In [ ]:
def read_metric_series(path):
    return pd.read_csv(path).set_index("metric")["value"]


def metric_number(metrics, name, default=0):
    if name not in metrics.index:
        return default
    value = pd.to_numeric(pd.Series([metrics.loc[name]]), errors="coerce").iloc[0]
    if pd.isna(value):
        return default
    return int(value) if float(value).is_integer() else float(value)


def latest_batch_rows(base_dir, filename_glob, *, stage_name=None):
    rows = []
    patterns = [f"*/batch_*/*/{filename_glob}"]
    if stage_name is not None:
        patterns.append(f"*/batch_*/{stage_name}/*/{filename_glob}")
    for pattern in patterns:
        for path in sorted(base_dir.glob(pattern)):
            parts = path.relative_to(base_dir).parts
            if len(parts) == 4:
                year_part, batch_part, runid_part, _ = parts
            elif len(parts) == 5:
                year_part, batch_part, stage_part, runid_part, _ = parts
                if stage_name is not None and stage_part != stage_name:
                    continue
            else:
                continue
            rows.append(
                {
                    "year": int(year_part),
                    "batch": batch_part,
                    "batch_number": int(batch_part.replace("batch_", "")),
                    "runid": runid_part,
                    "path": path,
                }
            )
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise FileNotFoundError(f"No files matching {filename_glob} found under {base_dir}")
    return (
        frame.sort_values(["year", "batch_number", "runid"])
        .drop_duplicates(["year", "batch_number"], keep="last")
        .reset_index(drop=True)
    )


def hours_per_million(elapsed_hours, wet_records_scanned):
    if wet_records_scanned == 0:
        return pd.NA
    return elapsed_hours / (wet_records_scanned / 1_000_000)

## 3. Trend Track

Trend counts come from the final processed trend file, because the processed builder enforces exactly one row per configured year. Throughput comes from the accepted per-year throughput summaries.

In [ ]:
trend_rates_path = PROCESSED_DIR / "trend/trend_rates.csv"
trend_rates = pd.read_csv(trend_rates_path)
trend_rates["year"] = trend_rates["year"].astype(int)
trend_reporting_rates = trend_rates.copy()
if "aggregation_level" in trend_reporting_rates.columns:
    trend_reporting_rates = trend_reporting_rates.loc[trend_reporting_rates["aggregation_level"].eq("all")].copy()

trend_metric_dir = INTERIM_COLLECTION_DIR / "metrics/trend"
trend_throughput_files = latest_batch_rows(trend_metric_dir, "cc_collection_throughput_summary_*.csv")

trend_throughput_records = []
for row in trend_throughput_files.itertuples(index=False):
    metrics = read_metric_series(row.path)
    trend_throughput_records.append(
        {
            "year": row.year,
            "batch_number": row.batch_number,
            "total_elapsed_hours": metric_number(metrics, "total_observed_elapsed_sec") / 3600,
            "wet_elapsed_hours": metric_number(metrics, "wet.elapsed_sec") / 3600,
            "warc_elapsed_hours": metric_number(metrics, "warc.elapsed_sec") / 3600,
            "document_quality_elapsed_hours": metric_number(metrics, "document_quality.elapsed_sec") / 3600,
            "warc_fetch_success_rate_pct": metric_number(metrics, "warc.fetch_success_rate_pct"),
            "warc_extract_success_rate_pct": metric_number(metrics, "warc.extract_success_rate_pct"),
        }
    )

trend_throughput = pd.DataFrame(trend_throughput_records)
trend_by_year = trend_reporting_rates.merge(trend_throughput, on="year", how="left")
trend_by_year = trend_by_year.rename(
    columns={
        "docs_scanned": "wet_records_scanned",
        "validated_hits_wet": "wet_validated_hits",
        "validated_hits_warc": "warc_validated_hits",
    }
)
trend_by_year["track"] = "trend"
trend_by_year["row_type"] = "year"
trend_by_year["batch_count"] = 1
trend_by_year["final_documents"] = pd.NA
trend_by_year["target_unique_documents"] = pd.NA
trend_by_year["adhd_membership_documents"] = pd.NA
trend_by_year["autism_membership_documents"] = pd.NA
trend_by_year["adhd_only_documents"] = pd.NA
trend_by_year["autism_only_documents"] = pd.NA
trend_by_year["adhd_autism_overlap_documents"] = pd.NA
trend_by_year["hours_per_million_wet_records"] = trend_by_year.apply(
    lambda row: hours_per_million(row["total_elapsed_hours"], row["wet_records_scanned"]), axis=1
)

trend_by_year[
    [
        "track",
        "year",
        "wet_records_scanned",
        "warc_validated_hits",
        "total_elapsed_hours",
        "hours_per_million_wet_records",
    ]
]

,track,year,wet_records_scanned,warc_validated_hits,total_elapsed_hours,hours_per_million_wet_records
0,trend,2014,5349060.0,13918.0,0.000000,0.000000
1,trend,2015,5291648.0,11898.0,2.133771,0.403234
2,trend,2016,5760599.0,19945.0,3.388940,0.588296
3,trend,2017,4435233.0,11828.0,1.878133,0.423458
4,trend,2018,4702862.0,12086.0,1.767904,0.375921
5,trend,2019,4377625.0,13847.0,2.457539,0.561386
6,trend,2020,5040987.0,14117.0,2.530095,0.501905
7,trend,2021,4818457.0,13996.0,2.806042,0.582353
8,trend,2022,4217647.0,11948.0,1.844984,0.437444
9,trend,2023,3806981.0,10102.0,1.460020,0.383511


## 4. Corpus Track

Corpus counts and throughput are read from the latest accepted summary for each year/batch. This remains valid after expansion batches are added.

In [ ]:
corpus_quality_dir = INTERIM_COLLECTION_DIR / "corpus"
corpus_metric_dir = INTERIM_COLLECTION_DIR / "metrics/corpus"
corpus_summary_files = latest_batch_rows(corpus_quality_dir, "cc_collection_summary_*.csv", stage_name="quality")
corpus_throughput_files = latest_batch_rows(corpus_metric_dir, "cc_collection_throughput_summary_*.csv")

corpus_records = []
for row in corpus_summary_files.itertuples(index=False):
    metrics = read_metric_series(row.path)
    corpus_records.append(
        {
            "year": row.year,
            "batch_number": row.batch_number,
            "runid": row.runid,
            "wet_records_scanned": metric_number(metrics, "docs_scanned"),
            "wet_validated_hits": metric_number(metrics, "validated_hits_wet"),
            "warc_validated_hits": metric_number(metrics, "validated_hits_warc"),
            "final_documents": metric_number(metrics, "final.doc_count"),
            "target_unique_documents": metric_number(metrics, "final.term_role.target.doc_count"),
            "adhd_membership_documents": metric_number(metrics, "final.term_group.adhd.doc_count"),
            "autism_membership_documents": metric_number(metrics, "final.term_group.autism.doc_count"),
        }
    )

corpus_throughput_records = []
for row in corpus_throughput_files.itertuples(index=False):
    metrics = read_metric_series(row.path)
    corpus_throughput_records.append(
        {
            "year": row.year,
            "batch_number": row.batch_number,
            "total_elapsed_hours": metric_number(metrics, "total_observed_elapsed_sec") / 3600,
            "wet_elapsed_hours": metric_number(metrics, "wet.elapsed_sec") / 3600,
            "warc_elapsed_hours": metric_number(metrics, "warc.elapsed_sec") / 3600,
            "document_quality_elapsed_hours": metric_number(metrics, "document_quality.elapsed_sec") / 3600,
            "warc_fetch_success_rate_pct": metric_number(metrics, "warc.fetch_success_rate_pct"),
            "warc_extract_success_rate_pct": metric_number(metrics, "warc.extract_success_rate_pct"),
        }
    )

corpus_batches = pd.DataFrame(corpus_records).merge(
    pd.DataFrame(corpus_throughput_records), on=["year", "batch_number"], how="left"
)
corpus_missing_throughput = corpus_batches.loc[
    corpus_batches["total_elapsed_hours"].isna(), ["year", "batch_number"]
].copy()

corpus_by_year = (
    corpus_batches.groupby("year", as_index=False)
    .agg(
        batch_count=("batch_number", "nunique"),
        wet_records_scanned=("wet_records_scanned", "sum"),
        wet_validated_hits=("wet_validated_hits", "sum"),
        warc_validated_hits=("warc_validated_hits", "sum"),
        final_documents=("final_documents", "sum"),
        target_unique_documents=("target_unique_documents", "sum"),
        adhd_membership_documents=("adhd_membership_documents", "sum"),
        autism_membership_documents=("autism_membership_documents", "sum"),
        throughput_batch_count=("total_elapsed_hours", "count"),
        total_elapsed_hours=("total_elapsed_hours", "sum"),
        wet_elapsed_hours=("wet_elapsed_hours", "sum"),
        warc_elapsed_hours=("warc_elapsed_hours", "sum"),
        document_quality_elapsed_hours=("document_quality_elapsed_hours", "sum"),
        warc_fetch_success_rate_pct=("warc_fetch_success_rate_pct", "min"),
        warc_extract_success_rate_pct=("warc_extract_success_rate_pct", "min"),
    )
)
corpus_by_year["track"] = "corpus"
corpus_by_year["row_type"] = "year"
corpus_by_year["throughput_complete"] = corpus_by_year["throughput_batch_count"].eq(
    corpus_by_year["batch_count"]
)
corpus_by_year["hours_per_million_wet_records"] = corpus_by_year.apply(
    lambda row: hours_per_million(row["total_elapsed_hours"], row["wet_records_scanned"])
    if row["throughput_complete"]
    else pd.NA,
    axis=1,
)
corpus_incomplete_throughput = ~corpus_by_year["throughput_complete"]
corpus_by_year.loc[
    corpus_incomplete_throughput,
    [
        "total_elapsed_hours",
        "wet_elapsed_hours",
        "warc_elapsed_hours",
        "document_quality_elapsed_hours",
        "warc_fetch_success_rate_pct",
        "warc_extract_success_rate_pct",
    ],
] = pd.NA

corpus_by_year[
    [
        "track",
        "year",
        "batch_count",
        "wet_records_scanned",
        "final_documents",
        "target_unique_documents",
        "total_elapsed_hours",
        "hours_per_million_wet_records",
    ]
]

,track,year,batch_count,wet_records_scanned,final_documents,target_unique_documents,total_elapsed_hours,hours_per_million_wet_records
0,corpus,2014,6,16312777,24123,6672,NaN,<NA>
1,corpus,2015,8,21279168,27537,7344,NaN,<NA>
2,corpus,2016,6,17442966,34603,8057,NaN,<NA>
3,corpus,2017,8,17750507,25425,6418,NaN,<NA>
4,corpus,2018,10,23615201,31496,8131,NaN,<NA>
5,corpus,2019,6,13124483,21414,6571,NaN,<NA>
6,corpus,2020,6,15020484,22084,5991,NaN,<NA>
7,corpus,2021,8,19289380,29262,7419,NaN,<NA>
8,corpus,2022,8,16911362,26933,6799,NaN,<NA>
9,corpus,2023,8,15238078,22550,5765,NaN,<NA>


## 5. Processed Corpus Check

The final corpus parquet is checked against the accepted batch summaries. This catches stale processed outputs after reruns or expansion batches.

In [ ]:
processed_corpus_path = PROCESSED_DIR / "corpus/corpus_documents.parquet"
processed_corpus = pd.read_parquet(
    processed_corpus_path,
    columns=["crawl_id", "url", "term_roles", "matched_terms", "source_year", "source_corpus_path"],
)

ADHD_TERM_NAMES = {"adhd", "attention_deficit"}
AUTISM_TERM_NAMES = {"autism", "autistic", "autism_spectrum", "asd"}

def pipe_membership(value):
    if pd.isna(value):
        return set()
    return {part.strip() for part in str(value).split("|") if part.strip()}

processed_duplicate_url_count = int(processed_corpus.duplicated(["crawl_id", "url"]).sum())
role_sets = processed_corpus["term_roles"].map(pipe_membership)
term_sets = processed_corpus["matched_terms"].map(pipe_membership)
processed_corpus["has_target_term"] = role_sets.map(lambda roles: "target" in roles)
processed_corpus["has_baseline_term"] = role_sets.map(lambda roles: "baseline" in roles)
processed_corpus["has_adhd_term"] = term_sets.map(lambda terms: bool(terms & ADHD_TERM_NAMES))
processed_corpus["has_autism_term"] = term_sets.map(lambda terms: bool(terms & AUTISM_TERM_NAMES))
processed_corpus["has_adhd_only"] = processed_corpus["has_adhd_term"] & ~processed_corpus["has_autism_term"]
processed_corpus["has_autism_only"] = ~processed_corpus["has_adhd_term"] & processed_corpus["has_autism_term"]
processed_corpus["has_adhd_autism_overlap"] = processed_corpus["has_adhd_term"] & processed_corpus["has_autism_term"]

processed_target_document_count = int(processed_corpus["has_target_term"].sum())
processed_baseline_document_count = int(processed_corpus["has_baseline_term"].sum())
processed_adhd_membership_count = int(processed_corpus["has_adhd_term"].sum())
processed_autism_membership_count = int(processed_corpus["has_autism_term"].sum())
processed_adhd_only_count = int(processed_corpus["has_adhd_only"].sum())
processed_autism_only_count = int(processed_corpus["has_autism_only"].sum())
processed_adhd_autism_overlap_count = int(processed_corpus["has_adhd_autism_overlap"].sum())
processed_source_count = int(processed_corpus["source_corpus_path"].nunique())

processed_target_membership_by_year = (
    processed_corpus.groupby("source_year", as_index=False)
    .agg(
        target_unique_documents=("has_target_term", "sum"),
        adhd_membership_documents=("has_adhd_term", "sum"),
        autism_membership_documents=("has_autism_term", "sum"),
        adhd_only_documents=("has_adhd_only", "sum"),
        autism_only_documents=("has_autism_only", "sum"),
        adhd_autism_overlap_documents=("has_adhd_autism_overlap", "sum"),
    )
    .rename(columns={"source_year": "year"})
)
processed_target_membership_by_year["year"] = processed_target_membership_by_year["year"].astype(int)

corpus_by_year = corpus_by_year.drop(
    columns=[
        "target_unique_documents",
        "adhd_membership_documents",
        "autism_membership_documents",
        "adhd_only_documents",
        "autism_only_documents",
        "adhd_autism_overlap_documents",
    ],
    errors="ignore",
).merge(processed_target_membership_by_year, on="year", how="left")

processed_summary = pd.DataFrame(
    [
        {
            "processed_rows": len(processed_corpus),
            "duplicate_crawl_url_rows": processed_duplicate_url_count,
            "duplicate_crawl_url_row_share": processed_duplicate_url_count / len(processed_corpus),
            "target_unique_documents": processed_target_document_count,
            "baseline_documents": processed_baseline_document_count,
            "adhd_membership_documents": processed_adhd_membership_count,
            "autism_membership_documents": processed_autism_membership_count,
            "adhd_only_documents": processed_adhd_only_count,
            "autism_only_documents": processed_autism_only_count,
            "adhd_autism_overlap_documents": processed_adhd_autism_overlap_count,
            "source_corpus_paths": processed_source_count,
        }
    ]
)

processed_summary

,processed_rows,duplicate_crawl_url_rows,duplicate_crawl_url_row_share,target_unique_documents,baseline_documents,adhd_membership_documents,autism_membership_documents,adhd_only_documents,autism_only_documents,adhd_autism_overlap_documents,source_corpus_paths
0,336178,258,0.000767,87173,258864,31354,67614,19559,55819,11795,108


## 6. Checks

These checks protect the paper numbers from partial syncs, stale processed outputs, and accidental duplicate years. Minor duplicate URL residue and missing timing summaries are reported as warnings rather than hard failures.

In [ ]:
trend_years = sorted(trend_reporting_rates["year"].tolist())
corpus_years = sorted(corpus_by_year["year"].tolist())
trend_duplicate_years = sorted(trend_reporting_rates.loc[trend_reporting_rates["year"].duplicated(), "year"].unique().tolist())
processed_duplicate_url_share = processed_duplicate_url_count / len(processed_corpus)

checks = pd.DataFrame(
    [
        {
            "check": "trend years match configured crawl map",
            "passed": trend_years == configured_years,
            "detail": f"found={trend_years}; expected={configured_years}",
        },
        {
            "check": "trend has no duplicate years",
            "passed": not trend_duplicate_years,
            "detail": f"duplicate_years={trend_duplicate_years}",
        },
        {
            "check": "corpus summaries cover every configured year",
            "passed": corpus_years == configured_years,
            "detail": f"found={corpus_years}; expected={configured_years}",
        },
        {
            "check": "corpus batch summaries match processed corpus sources",
            "passed": len(corpus_batches) == processed_source_count,
            "detail": f"summary_batches={len(corpus_summary_files)}; processed_sources={processed_source_count}",
        },
        {
            "check": "processed corpus row count matches accepted quality summaries",
            "passed": len(processed_corpus) == int(corpus_by_year["final_documents"].sum()),
            "detail": f"processed={len(processed_corpus)}; summaries={int(corpus_by_year['final_documents'].sum())}",
        },
        {
            "check": "processed target document count matches accepted quality summaries",
            "passed": processed_target_document_count == int(corpus_by_year["target_unique_documents"].sum()),
            "detail": f"processed={processed_target_document_count}; summaries={int(corpus_by_year['target_unique_documents'].sum())}",
        },
        {
            "check": "processed target union equals ADHD/Autism membership union",
            "passed": processed_target_document_count == processed_adhd_only_count + processed_autism_only_count + processed_adhd_autism_overlap_count,
            "detail": f"target_unique={processed_target_document_count}; adhd_only={processed_adhd_only_count}; autism_only={processed_autism_only_count}; overlap={processed_adhd_autism_overlap_count}",
        },
    ]
)

if not checks["passed"].all():
    display(checks)
    failed = checks.loc[~checks["passed"], "check"].tolist()
    raise AssertionError(f"Collection reporting checks failed: {failed}")

warnings = pd.DataFrame(
    [
        {
            "warning": "trend throughput coverage",
            "active": sorted(trend_throughput["year"].tolist()) != configured_years,
            "detail": f"found={sorted(trend_throughput['year'].tolist())}; expected={configured_years}",
        },
        {
            "warning": "corpus throughput coverage",
            "active": not corpus_missing_throughput.empty,
            "detail": f"missing_batch_timings={corpus_missing_throughput.to_dict('records')}",
        },
        {
            "warning": "processed corpus duplicate crawl_id/url residue",
            "active": processed_duplicate_url_count > 0,
            "detail": f"duplicate_rows={processed_duplicate_url_count}; share={processed_duplicate_url_share:.6%}",
        },
    ]
)

if warnings["active"].any():
    display(warnings.loc[warnings["active"]])

checks

,warning,active,detail
0,trend throughput coverage,True,"found=[2014, 2015, 2016, 2017, 2018, 2019, 202..."
1,corpus throughput coverage,True,"missing_batch_timings=[{'year': 2014, 'batch_n..."
2,processed corpus duplicate crawl_id/url residue,True,duplicate_rows=258; share=0.076745%


,check,passed,detail
0,trend years match configured crawl map,True,"found=[2014, 2015, 2016, 2017, 2018, 2019, 202..."
1,trend has no duplicate years,True,duplicate_years=[]
2,corpus summaries cover every configured year,True,"found=[2014, 2015, 2016, 2017, 2018, 2019, 202..."
3,corpus batch summaries match processed corpus ...,True,summary_batches=108; processed_sources=108
4,processed corpus row count matches accepted qu...,True,processed=336178; summaries=336178
5,processed target document count matches accept...,True,processed=87173; summaries=87173
6,processed target union equals ADHD/Autism memb...,True,target_unique=87173; adhd_only=19559; autism_o...


## 7. Single Reporting CSV

The CSV contains one topline row per track and one annual row per track/year. The normalized throughput measure is `hours_per_million_wet_records`, computed from total observed elapsed time divided by WET records scanned. This is the number to use for a concise runtime statement.

In [ ]:
report_columns = [
    "row_type",
    "track",
    "year",
    "year_count",
    "year_range",
    "batch_count",
    "instance_type",
    "wet_records_scanned",
    "wet_validated_hits",
    "warc_validated_hits",
    "final_documents",
    "target_unique_documents",
    "adhd_membership_documents",
    "autism_membership_documents",
    "adhd_only_documents",
    "autism_only_documents",
    "adhd_autism_overlap_documents",
    "total_elapsed_hours",
    "hours_per_million_wet_records",
    "throughput_observed_wet_records_scanned",
    "throughput_observed_year_count",
    "throughput_complete",
    "wet_elapsed_hours",
    "warc_elapsed_hours",
    "document_quality_elapsed_hours",
    "warc_fetch_success_rate_pct",
    "warc_extract_success_rate_pct",
    "report_note",
]

year_rows = pd.concat([trend_by_year, corpus_by_year], ignore_index=True)
year_rows["year_count"] = pd.NA
year_rows["year_range"] = pd.NA
year_rows["instance_type"] = instance_type
year_rows["throughput_complete"] = year_rows["throughput_complete"].where(
    year_rows["throughput_complete"].notna(), year_rows["total_elapsed_hours"].notna()
).astype(bool)
year_rows["throughput_observed_year_count"] = year_rows["throughput_complete"].astype(int)
year_rows["throughput_observed_wet_records_scanned"] = year_rows["wet_records_scanned"].where(
    year_rows["throughput_complete"], pd.NA
)
year_rows["report_note"] = "Annual accepted collection output."

topline_rows = []
for track, frame in year_rows.groupby("track", sort=False):
    total_wet = int(frame["wet_records_scanned"].sum())
    throughput_frame = frame.loc[frame["throughput_complete"]].copy()
    observed_wet = int(throughput_frame["wet_records_scanned"].sum())
    observed_year_count = int(throughput_frame["year"].nunique())
    throughput_complete = observed_year_count == year_count
    total_elapsed = float(throughput_frame["total_elapsed_hours"].sum())
    topline_rows.append(
        {
            "row_type": "track_total",
            "track": track,
            "year": pd.NA,
            "year_count": year_count,
            "year_range": f"{year_min}-{year_max}",
            "batch_count": int(frame["batch_count"].sum()),
            "instance_type": instance_type,
            "wet_records_scanned": total_wet,
            "wet_validated_hits": int(frame["wet_validated_hits"].sum()),
            "warc_validated_hits": int(frame["warc_validated_hits"].sum()),
            "final_documents": int(frame["final_documents"].sum()) if track == "corpus" else pd.NA,
            "target_unique_documents": int(frame["target_unique_documents"].sum()) if track == "corpus" else pd.NA,
            "adhd_membership_documents": int(frame["adhd_membership_documents"].sum()) if track == "corpus" else pd.NA,
            "autism_membership_documents": int(frame["autism_membership_documents"].sum()) if track == "corpus" else pd.NA,
            "adhd_only_documents": int(frame["adhd_only_documents"].sum()) if track == "corpus" else pd.NA,
            "autism_only_documents": int(frame["autism_only_documents"].sum()) if track == "corpus" else pd.NA,
            "adhd_autism_overlap_documents": int(frame["adhd_autism_overlap_documents"].sum()) if track == "corpus" else pd.NA,
            "total_elapsed_hours": total_elapsed,
            "hours_per_million_wet_records": hours_per_million(total_elapsed, observed_wet),
            "throughput_observed_wet_records_scanned": observed_wet,
            "throughput_observed_year_count": observed_year_count,
            "throughput_complete": throughput_complete,
            "wet_elapsed_hours": float(throughput_frame["wet_elapsed_hours"].sum()),
            "warc_elapsed_hours": float(throughput_frame["warc_elapsed_hours"].sum()),
            "document_quality_elapsed_hours": float(throughput_frame["document_quality_elapsed_hours"].sum()),
            "warc_fetch_success_rate_pct": float(throughput_frame["warc_fetch_success_rate_pct"].min()),
            "warc_extract_success_rate_pct": float(throughput_frame["warc_extract_success_rate_pct"].min()),
            "report_note": "Fixed-effort annual trend track." if track == "trend" else "Quality-gated corpus track.",
        }
    )

collection_reporting = pd.concat([pd.DataFrame(topline_rows), year_rows], ignore_index=True)
collection_reporting = collection_reporting[report_columns].sort_values(
    ["track", "row_type", "year"], na_position="first"
)

integer_columns = [
    "year",
    "year_count",
    "batch_count",
    "wet_records_scanned",
    "wet_validated_hits",
    "warc_validated_hits",
    "final_documents",
    "target_unique_documents",
    "adhd_membership_documents",
    "autism_membership_documents",
    "adhd_only_documents",
    "autism_only_documents",
    "adhd_autism_overlap_documents",
    "throughput_observed_wet_records_scanned",
    "throughput_observed_year_count",
]
for column in integer_columns:
    collection_reporting[column] = pd.to_numeric(collection_reporting[column], errors="coerce").astype("Int64")

float_columns = [
    "total_elapsed_hours",
    "hours_per_million_wet_records",
    "wet_elapsed_hours",
    "warc_elapsed_hours",
    "document_quality_elapsed_hours",
    "warc_fetch_success_rate_pct",
    "warc_extract_success_rate_pct",
]
for column in float_columns:
    collection_reporting[column] = pd.to_numeric(collection_reporting[column], errors="coerce").round(3)

REPORTING_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
collection_reporting.to_csv(REPORTING_CSV_PATH, index=False)

collection_reporting

,row_type,track,year,year_count,year_range,batch_count,instance_type,wet_records_scanned,wet_validated_hits,warc_validated_hits,final_documents,target_unique_documents,adhd_membership_documents,autism_membership_documents,adhd_only_documents,autism_only_documents,adhd_autism_overlap_documents,total_elapsed_hours,hours_per_million_wet_records,throughput_observed_wet_records_scanned,throughput_observed_year_count,throughput_complete,wet_elapsed_hours,warc_elapsed_hours,document_quality_elapsed_hours,warc_fetch_success_rate_pct,warc_extract_success_rate_pct,report_note
1,track_total,corpus,<NA>,13,2014-2026,108,m7i-flex.large,220017831,1354386,632110,336178,87173,31354,67614,19559,55819,11795,0.000,NaN,0,0,False,0.000,0.000,0.000,NaN,NaN,Quality-gated corpus track.
15,year,corpus,2014,<NA>,<NA>,6,m7i-flex.large,16312777,100144,45220,24123,6672,2248,5210,1462,4424,786,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
16,year,corpus,2015,<NA>,<NA>,8,m7i-flex.large,21279168,114891,52060,27537,7344,2307,5851,1493,5037,814,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
17,year,corpus,2016,<NA>,<NA>,6,m7i-flex.large,17442966,135732,64432,34603,8057,2495,6417,1640,5562,855,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
18,year,corpus,2017,<NA>,<NA>,8,m7i-flex.large,17750507,106667,48608,25425,6418,2130,5137,1281,4288,849,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
19,year,corpus,2018,<NA>,<NA>,10,m7i-flex.large,23615201,140547,62478,31496,8131,2685,6425,1706,5446,979,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
20,year,corpus,2019,<NA>,<NA>,6,m7i-flex.large,13124483,85640,41843,21414,6571,2598,5216,1355,3973,1243,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
21,year,corpus,2020,<NA>,<NA>,6,m7i-flex.large,15020484,100738,43639,22084,5991,2218,4698,1293,3773,925,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
22,year,corpus,2021,<NA>,<NA>,8,m7i-flex.large,19289380,125829,56176,29262,7419,2704,5722,1697,4715,1007,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
23,year,corpus,2022,<NA>,<NA>,8,m7i-flex.large,16911362,107091,48710,26933,6799,2643,5088,1711,4156,932,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
